In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# 2. Clone YOLOv5 Official Repository
# ============================================================
%cd /content

!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5.git

%cd /content/yolov5
!pip install -r requirements.txt -q

/content
Cloning into 'yolov5'...
remote: Enumerating objects: 18359, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 18359 (delta 33), reused 17 (delta 17), pack-reused 18307 (from 2)
Receiving objects: 100% (18359/18359), 17.47 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (12478/12478), done.
/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 3. Unzip Dataset Helmet Detection YOLOv5 PyTorch
# ============================================================
import os

!rm -rf /content/dataset
!mkdir -p /content/dataset

# GANTI SESUAI NAMA FILE ZIP DATASET KAMU DI GOOGLE DRIVE
DATASET_ZIP = "/content/drive/MyDrive/helmet detection.v2i.yolov5pytorch.zip"

!unzip -q "{DATASET_ZIP}" -d /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [ ]:
# ============================================================
# 4. Cari data.yaml Otomatis
# ============================================================
import os

dataset_path = None

for root, dirs, files in os.walk("/content/dataset"):
    if "data.yaml" in files:
        dataset_path = root
        break

if dataset_path is None:
    raise FileNotFoundError("data.yaml tidak ditemukan. Cek lagi file zip dataset.")

print("Dataset path:", dataset_path)
print("Isi folder dataset:", os.listdir(dataset_path))

Dataset path: /content/dataset
Isi folder dataset: ['data.yaml', 'README.dataset.txt', 'valid', 'train', 'README.roboflow.txt', 'test']


In [ ]:
# ============================================================
# 5. Baca data.yaml
# ============================================================
import yaml
import os

yaml_path = os.path.join(dataset_path, "data.yaml")

with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print("Data YAML awal:")
print(data_yaml)

Data YAML awal:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 6. Fix Path data.yaml untuk Colab
# ============================================================
import os
import yaml

# Roboflow biasanya pakai train/valid/test
train_dir = os.path.join(dataset_path, "train/images")
valid_dir = os.path.join(dataset_path, "valid/images")
val_dir   = os.path.join(dataset_path, "val/images")
test_dir  = os.path.join(dataset_path, "test/images")

data_yaml["train"] = train_dir

if os.path.exists(valid_dir):
    data_yaml["val"] = valid_dir
elif os.path.exists(val_dir):
    data_yaml["val"] = val_dir
else:
    raise FileNotFoundError("Folder valid/images atau val/images tidak ditemukan.")

if os.path.exists(test_dir):
    data_yaml["test"] = test_dir
else:
    print("Peringatan: folder test/images tidak ditemukan. Evaluasi test nanti bisa gagal.")

# Rapikan format names
if "names" in data_yaml:
    names = data_yaml["names"]
    if isinstance(names, dict):
        names = [names[i] for i in range(len(names))]
    data_yaml["names"] = names
    data_yaml["nc"] = len(names)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Data YAML setelah diperbaiki:")
print(data_yaml)

Data YAML setelah diperbaiki:
{'train': '/content/dataset/train/images', 'val': '/content/dataset/valid/images', 'test': '/content/dataset/test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 7. Cek Jumlah Dataset
# ============================================================
from pathlib import Path

def get_split_folder(split):
    if split == "valid":
        if (Path(dataset_path) / "valid").exists():
            return "valid"
        elif (Path(dataset_path) / "val").exists():
            return "val"
    return split

def count_dataset(split):
    split_folder = get_split_folder(split)

    img_dir = Path(dataset_path) / split_folder / "images"
    label_dir = Path(dataset_path) / split_folder / "labels"

    images = list(img_dir.glob("*")) if img_dir.exists() else []
    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

    print(f"{split_folder}")
    print("Images :", len(images))
    print("Labels :", len(labels))
    print("-" * 30)

for split in ["train", "valid", "test"]:
    count_dataset(split)

train
Images : 6482
Labels : 6482
------------------------------
valid
Images : 1671
Labels : 1671
------------------------------
test
Images : 869
Labels : 869
------------------------------


In [ ]:
# ============================================================
# 8. Cleaning Dataset
# Hapus label kosong, gambar tanpa label, dan label tanpa gambar
# ============================================================
from pathlib import Path

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

def clean_dataset(split):
    split_folder = get_split_folder(split)

    img_dir = Path(dataset_path) / split_folder / "images"
    label_dir = Path(dataset_path) / split_folder / "labels"

    if not img_dir.exists() or not label_dir.exists():
        print(f"Split {split_folder} tidak ditemukan, dilewati.")
        return

    print(f"\nMembersihkan split: {split_folder}")

    # Hapus label kosong
    empty_count = 0
    for label_path in label_dir.glob("*.txt"):
        if label_path.stat().st_size == 0:
            label_path.unlink()
            empty_count += 1

    # Hapus gambar tanpa label
    removed_images = 0
    for img_path in img_dir.iterdir():
        if img_path.suffix in IMG_EXTS:
            label_path = label_dir / (img_path.stem + ".txt")
            if not label_path.exists():
                img_path.unlink()
                removed_images += 1

    # Hapus label tanpa gambar
    image_stems = {p.stem for p in img_dir.iterdir() if p.suffix in IMG_EXTS}
    removed_labels = 0

    for label_path in label_dir.glob("*.txt"):
        if label_path.stem not in image_stems:
            label_path.unlink()
            removed_labels += 1

    print("Label kosong dihapus:", empty_count)
    print("Gambar tanpa label dihapus:", removed_images)
    print("Label tanpa gambar dihapus:", removed_labels)

for split in ["train", "valid", "test"]:
    clean_dataset(split)


Membersihkan split: train
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: valid
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: test
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0


In [ ]:
# ============================================================
# 9. Validasi Format Label YOLO
# Format benar: class x_center y_center width height
# ============================================================
from pathlib import Path

NC = data_yaml["nc"]
print("Jumlah kelas:", NC)
print("Nama kelas:", data_yaml["names"])

def validate_labels(split):
    split_folder = get_split_folder(split)
    label_dir = Path(dataset_path) / split_folder / "labels"

    if not label_dir.exists():
        print(f"{split_folder} labels tidak ditemukan, dilewati.")
        return []

    bad_files = []

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):
            parts = line.strip().split()

            if len(parts) != 5:
                bad_files.append((label_path.name, line_number, "jumlah kolom bukan 5", line.strip()))
                break

            try:
                cls = int(float(parts[0]))
                values = list(map(float, parts[1:]))
            except:
                bad_files.append((label_path.name, line_number, "format angka salah", line.strip()))
                break

            if cls < 0 or cls >= NC:
                bad_files.append((label_path.name, line_number, f"class id salah: {cls}", line.strip()))
                break

            if any(v < 0 or v > 1 for v in values):
                bad_files.append((label_path.name, line_number, "koordinat di luar 0-1", line.strip()))
                break

    print(f"{split_folder} label bermasalah:", len(bad_files))

    for item in bad_files[:10]:
        print(item)

    return bad_files

all_bad = []

for split in ["train", "valid", "test"]:
    all_bad.extend(validate_labels(split))

print("Total semua label bermasalah:", len(all_bad))

Jumlah kelas: 3
Nama kelas: ['helmet', 'no_helmet', 'person']
train label bermasalah: 0
valid label bermasalah: 0
test label bermasalah: 1
('005377_jpg.rf.877294944cd6d730dd900feb00280ffe.txt', 2, 'jumlah kolom bukan 5', '0 0.5925 0.975 0.5900000000000001 0.9016666666666666 0.5775 0.8783333333333333 0.5575 0.7183333333333334 0.60375 0.7 0.6375 0.8816666666666666 0.675 0.8716666666666667 0.6625 0.7016666666666667 0.665 0.6716666666666666 0.7025 0.5916666666666667 0.7125 0.3983333333333333 0.7025 0.335 0.6625 0.2783333333333333 0.65 0.23166666666666663 0.6525000000000001 0.19833333333333333 0.6162500000000001 0.16 0.58875 0.16666666666666669 0.5650000000000001 0.19166666666666668 0.5625 0.24833333333333335 0.5650000000000001 0.2783333333333333 0.5825 0.29166666666666663 0.60375 0.33666666666666667 0.6225 0.3416666666666667 0.62 0.3783333333333333 0.60875 0.4066666666666666 0.5537500000000001 0.4 0.535 0.4116666666666666 0.5375 0.43166666666666664 0.5925 0.505 0.5900000000000001 0.5916666

In [ ]:
# ============================================================
# 9.1 Konversi Polygon ke Bounding Box jika ada label polygon
# Jalankan ini jika validasi menemukan "jumlah kolom bukan 5"
# Kalau label sudah benar, kode ini tetap aman.
# ============================================================
from pathlib import Path

def convert_label_to_bbox(label_path):
    new_lines = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 5:
            continue

        cls = parts[0]
        coords = list(map(float, parts[1:]))

        # Kalau sudah bbox YOLO
        if len(coords) == 4:
            x_center, y_center, w, h = coords

        # Kalau polygon
        else:
            if len(coords) % 2 != 0:
                continue

            xs = coords[0::2]
            ys = coords[1::2]

            x_min = min(xs)
            x_max = max(xs)
            y_min = min(ys)
            y_max = max(ys)

            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            w = x_max - x_min
            h = y_max - y_min

        # Batasi nilai ke 0 sampai 1
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        w = max(0, min(1, w))
        h = max(0, min(1, h))

        if w <= 0 or h <= 0:
            continue

        new_lines.append(f"{cls} {x_center} {y_center} {w} {h}")

    with open(label_path, "w") as f:
        f.write("\n".join(new_lines))


def convert_all_labels():
    for split in ["train", "valid", "test"]:
        split_folder = get_split_folder(split)
        label_dir = Path(dataset_path) / split_folder / "labels"

        if not label_dir.exists():
            continue

        count = 0
        for label_path in label_dir.glob("*.txt"):
            convert_label_to_bbox(label_path)
            count += 1

        print(f"{split_folder}: {count} file label diproses")

convert_all_labels()

In [ ]:
# ============================================================
# 10. Training YOLOv5s Helmet Detection
# ============================================================
import time

%cd /content/yolov5

start_train = time.time()

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data "{yaml_path}" \
  --weights yolov5s.pt \
  --device 0 \
  --workers 2 \
  --patience 30 \
  --project /content/helmet_detection_yolov5 \
  --name yolov5s_helmet_detection_img640 \
  --exist-ok \
  --cos-lr

end_train = time.time()

training_time_seconds = end_train - start_train
training_time_minutes = training_time_seconds / 60
training_time_hours = training_time_seconds / 3600

print("Training time:", training_time_seconds, "detik")
print("Training time:", training_time_minutes, "menit")
print("Training time:", training_time_hours, "jam")

Streaming output truncated to the last 5000 lines.
      93/99      4.35G    0.02414    0.02135   0.001231         95        640:  90% 367/406 [02:33<00:17,  2.27it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      93/99      4.35G    0.02414    0.02135    0.00123         98        640:  91% 368/406 [02:33<00:16,  2.24it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      93/99      4.35G    0.02414    0.02134   0.001229         76        640:  91% 369/406 [02:33<00:14,  2.59it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      93/99      4.35G    0.024

In [ ]:
# ============================================================
# 11. Evaluasi Best Model YOLOv5s pada Test Set
# ============================================================
import os
import sys

%cd /content/yolov5

best_model_path = "/content/helmet_detection_yolov5/yolov5s_helmet_detection_img640/weights/best.pt"

if not os.path.exists(best_model_path):
    raise FileNotFoundError("best.pt tidak ditemukan. Pastikan training YOLOv5s sudah selesai.")

sys.path.append("/content/yolov5")

from val import run

results, maps, times = run(
    data=yaml_path,
    weights=best_model_path,
    imgsz=640,
    batch_size=8,
    task="test",
    device="0",
    project="/content/helmet_detection_eval",
    name="final_test_yolov5s_img640",
    exist_ok=True,
    plots=True
)

precision = float(results[0])
recall = float(results[1])
map50 = float(results[2])
map50_95 = float(results[3])
f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)

print("Precision :", precision)
print("Recall    :", recall)
print("F1-score  :", f1_score)
print("mAP@50    :", map50)
print("mAP@50:95 :", map50_95)
print("Per-class mAP@50:95:", maps)
print("Speed:", times)

/content/yolov5


YOLOv5 🚀 v7.0-503-gb8493964 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /content/dataset/test/labels... 869 images, 0 backgrounds, 0 corrupt: 100%|██████████| 869/869 [00:01<00:00, 552.33it/s]
test: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100%|██████████| 109/109 [00:13<00:00,  8.28it/s]
                   all        869       3271      0.824      0.803      0.834      0.551
                helmet        869       2110      0.912      0.923      0.946      0.635
             no_helmet        869        743      0.913      0.913       0.95      0.667
                person        869        418      0.647      0.574      0.605      0.352
Speed: 0.3ms pre-process, 3.9ms inference, 2.0ms NMS per image at shape (8, 3, 640, 640)
Results saved to /content/helmet_detec

Precision : 0.8239257721848801
Recall    : 0.8034553913404792
F1-score  : 0.8135618356125979
mAP@50    : 0.8335454754605153
mAP@50:95 : 0.5514993621785255
Per-class mAP@50:95: [    0.63519     0.66748     0.35183]
Speed: (0.2827627886562545, 3.945976735807535, 1.9998180029444372)


In [ ]:
# ============================================================
# 12. Hitung Inference Time dan FPS YOLOv5s
# ============================================================
import torch
import glob
import os
import time

TEST_IMG_DIR = os.path.join(dataset_path, "test/images")

test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(glob.glob(os.path.join(TEST_IMG_DIR, ext)))

print("Jumlah gambar test:", len(test_images))

model_v5 = torch.hub.load(
    "/content/yolov5",
    "custom",
    path=best_model_path,
    source="local"
)

if torch.cuda.is_available():
    model_v5.to("cuda")

model_v5.conf = 0.25

# Warm up
for img_path in test_images[:10]:
    _ = model_v5(img_path, size=640)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.time()

for img_path in test_images:
    _ = model_v5(img_path, size=640)

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.time()

total_inference_time = end_time - start_time
avg_inference_time = total_inference_time / len(test_images)
fps = 1 / avg_inference_time

print("Total inference time:", total_inference_time, "detik")
print("Average inference time:", avg_inference_time, "detik/gambar")
print("FPS:", fps)

YOLOv5 🚀 v7.0-503-gb8493964 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 


Jumlah gambar test: 869


Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda

Total inference time: 10.454102277755737 detik
Average inference time: 0.012030037143562414 detik/gambar
FPS: 83.12526287877058


/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:931: Fu

In [ ]:
# ============================================================
# 13. Simpan Metrik YOLOv5s ke CSV
# ============================================================
import pandas as pd
import os

OUTPUT_DIR = "/content/helmet_detection_eval/yolov5s_final_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary = pd.DataFrame([{
    "Model": "YOLOv5s",
    "Dataset": "Helmet Detection",
    "Jumlah Kelas": data_yaml["nc"],
    "Kelas": ", ".join(data_yaml["names"]),
    "Epochs": 100,
    "Image Size": 640,
    "Batch": 16,
    "Training Time (s)": training_time_seconds,
    "Training Time (minutes)": training_time_minutes,
    "Training Time (hours)": training_time_hours,
    "Accuracy": "",
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1_score,
    "mAP@50": map50,
    "mAP@50:95": map50_95,
    "Inference Time per Image (s)": avg_inference_time,
    "FPS": fps
}])

summary_path = os.path.join(OUTPUT_DIR, "yolov5s_final_metrics.csv")
summary.to_csv(summary_path, index=False)

summary

,Model,Dataset,Jumlah Kelas,Kelas,Epochs,...,F1-Score,mAP@50,mAP@50:95,Inference Time per Image (s),FPS
0,YOLOv5s,Helmet Detection,3,"helmet, no_helmet, person",100,...,0.813562,0.833545,0.551499,0.01203,83.125263


In [ ]:
# ============================================================
# 13.1 Simpan Per-Class mAP@50:95 ke CSV
# ============================================================
class_names = data_yaml["names"]

per_class_rows = []

for i, class_name in enumerate(class_names):
    per_class_rows.append({
        "Model": "YOLOv5s",
        "Class": class_name,
        "mAP@50:95": float(maps[i]) if i < len(maps) else None
    })

per_class_df = pd.DataFrame(per_class_rows)

per_class_path = os.path.join(OUTPUT_DIR, "yolov5s_per_class_metrics.csv")
per_class_df.to_csv(per_class_path, index=False)

per_class_df

,Model,Class,mAP@50:95
0,YOLOv5s,helmet,0.635191
1,YOLOv5s,no_helmet,0.667477
2,YOLOv5s,person,0.351830


In [ ]:
# ============================================================
# 14. Simpan Sample Prediction YOLOv5s
# ============================================================
%cd /content/yolov5

!python detect.py \
  --weights "{best_model_path}" \
  --source "{TEST_IMG_DIR}" \
  --img 640 \
  --conf 0.25 \
  --project /content/helmet_detection_eval \
  --name yolov5s_prediction_sample \
  --exist-ok

/content/yolov5
detect: weights=['/content/helmet_detection_yolov5/yolov5s_helmet_detection_img640/weights/best.pt'], source=/content/dataset/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/helmet_detection_eval, name=yolov5s_prediction_sample, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-503-gb8493964 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
image 1/869 /content/dataset/test/images/005298_jpg.rf.9095ddb4a98d41155cd939563545e6a4.jpg: 640x640 7 helmets, 9.9ms
image 2/869 /content/dataset/test/images/005299_jpg.rf.28828b9181484ee4badd52f95

In [ ]:
# ============================================================
# 15. Kumpulkan Semua Hasil YOLOv5s ke Folder Final
# ============================================================
import os
import shutil
import glob

RESULT_FOLDER = "/content/YOLOv5s_Helmet_Detection_Final_Result"

if os.path.exists(RESULT_FOLDER):
    shutil.rmtree(RESULT_FOLDER)

os.makedirs(RESULT_FOLDER, exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/01_Model", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/02_Training_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/03_Evaluation_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/04_Visualization", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/05_Prediction_Sample", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/06_Config", exist_ok=True)

TRAIN_DIR = "/content/helmet_detection_yolov5/yolov5s_helmet_detection_img640"
EVAL_DIR = "/content/helmet_detection_eval/final_test_yolov5s_img640"
PRED_DIR = "/content/helmet_detection_eval/yolov5s_prediction_sample"

# Copy model
if os.path.exists(f"{TRAIN_DIR}/weights/best.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/best.pt", f"{RESULT_FOLDER}/01_Model/best_yolov5s.pt")

if os.path.exists(f"{TRAIN_DIR}/weights/last.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/last.pt", f"{RESULT_FOLDER}/01_Model/last_yolov5s.pt")

# Copy hasil training
for file in glob.glob(f"{TRAIN_DIR}/*"):
    if os.path.isfile(file):
        shutil.copy(file, f"{RESULT_FOLDER}/02_Training_Result/{os.path.basename(file)}")

# Copy hasil evaluasi CSV
if os.path.exists(OUTPUT_DIR):
    for file in glob.glob(f"{OUTPUT_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy hasil evaluasi test
if os.path.exists(EVAL_DIR):
    for file in glob.glob(f"{EVAL_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy visualisasi training
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    for file in glob.glob(f"{TRAIN_DIR}/{ext}"):
        shutil.copy(file, f"{RESULT_FOLDER}/04_Visualization/{os.path.basename(file)}")

# Copy prediction sample
if os.path.exists(PRED_DIR):
    for file in glob.glob(f"{PRED_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/05_Prediction_Sample/{os.path.basename(file)}")

# Copy config
shutil.copy(yaml_path, f"{RESULT_FOLDER}/06_Config/data.yaml")

# README
readme = f"""
YOLOv5s Helmet Detection Final Result

Judul:
Deteksi Kepatuhan Penggunaan Helm Keselamatan Menggunakan Perbandingan YOLOv5, YOLOv8, dan YOLOv11 pada Dataset Publik

Model:
YOLOv5s

Dataset:
Helmet Detection

Class:
{", ".join(data_yaml["names"])}

Setting Training:
- Epochs: 100
- Image Size: 640
- Batch: 16
- Patience: 30
- cos_lr: True
- Framework: Official YOLOv5 PyTorch Repository

Metrik:
- Precision
- Recall
- F1-score
- mAP@50
- mAP@50:95
- Inference Time
- FPS
- Confusion Matrix
- Per-Class mAP
"""

with open(f"{RESULT_FOLDER}/README.txt", "w") as f:
    f.write(readme)

print("Folder hasil dibuat:", RESULT_FOLDER)

Folder hasil dibuat: /content/YOLOv5s_Helmet_Detection_Final_Result


In [ ]:
# ============================================================
# 16. Simpan Folder Final ke Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TARGET = "/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv5s_Final_Result"

if os.path.exists(DRIVE_TARGET):
    shutil.rmtree(DRIVE_TARGET)

shutil.copytree(RESULT_FOLDER, DRIVE_TARGET)

print("Hasil YOLOv5s Helmet Detection sudah masuk ke Google Drive:")
print(DRIVE_TARGET)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hasil YOLOv5s Helmet Detection sudah masuk ke Google Drive:
/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv5s_Final_Result


In [ ]:
# ============================================================
# 17. Buat ZIP dan Download
# ============================================================
import shutil
from google.colab import files

shutil.make_archive("/content/YOLOv5s_Helmet_Detection_Final_Result", "zip", RESULT_FOLDER)

print("ZIP dibuat di: /content/YOLOv5s_Helmet_Detection_Final_Result.zip")

files.download("/content/YOLOv5s_Helmet_Detection_Final_Result.zip")